# Notebook 2 — Other distributions, same reading

**Companion to F-004, *Probability as a Time Series*.**

Notebook 1 showed the trajectory reading on the fair coin. This notebook applies the *same* machinery to four other primitive cases:

- a biased coin (Binomial with $p \neq 0.5$),
- a Gaussian random walk,
- a Cauchy random walk (heavy-tail step, no moments),
- an AR(1) Gaussian process (memory).

The same operator $\hat M$ and the same residual cloud read the substantive content in every case.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def fit_M(Q):
    X, Yt = Q[:-1], Q[1:]
    G = X.T @ X + 1e-8 * np.eye(3)
    return np.linalg.solve(G, X.T @ Yt).T

def make_jet(Y):
    """Backward-difference kinematic jet. Drop first 3 boundary samples."""
    Y = np.asarray(Y, dtype=float)
    dY  = np.zeros_like(Y); dY[1:]  = Y[1:]  - Y[:-1]
    ddY = np.zeros_like(Y); ddY[2:] = dY[2:] - dY[1:-1]
    return np.column_stack([Y, dY, ddY])[3:]

# Establish empirical M_iid on a long fair-coin trajectory (Notebook 1)
rng0 = np.random.default_rng(42)
fair_steps = rng0.choice([-1, 1], size=20000)
Y_fair = np.cumsum(fair_steps).astype(float)
M_iid = fit_M(make_jet(Y_fair))
print('Empirical M_iid (fair coin, N=20000):')
print(np.round(M_iid, 4))
print(f'\nscalar trace s_0 = {np.trace(M_iid)/3:.4f}  (should be ~1/3)')

n = 5000


## 1.  Biased coin, $p = 0.3$

Now the heads-frequency limit is $0.3$, not $0.5$. The marginal of the increment is different, but the operator should still land at $\hat M_{\mathrm{iid}}$.


In [ ]:
rng = np.random.default_rng(1)
steps_b = rng.choice([1, -1], size=n, p=[0.3, 0.7])
Y_b = np.cumsum(steps_b).astype(float)
Q_b = make_jet(Y_b)
M_b = fit_M(Q_b)
print('biased coin (p=0.3), M_hat:')
print(np.round(M_b, 4))
print(f'||M_b - M_iid||_F = {np.linalg.norm(M_b - M_iid):.5f}')


## 2.  Gaussian random walk

The step is now $X_t \sim \mathcal{N}(0, 1)$. The marginal of $Y_n = X_1 + \cdots + X_n$ is $\mathcal{N}(0, n)$.


In [ ]:
z = rng.standard_normal(n)
Y_g = np.cumsum(z)
Q_g = make_jet(Y_g)
M_g = fit_M(Q_g)
print('Gaussian random walk, M_hat:')
print(np.round(M_g, 4))
print(f'||M_g - M_iid||_F = {np.linalg.norm(M_g - M_iid):.5f}')


## 3.  Cauchy random walk

The step is $X_t \sim \mathrm{Cauchy}(0, 1)$. No moments exist. The classical Central Limit reading breaks. But the operator should still land at $\hat M_{\mathrm{iid}}$, because the dynamics are still i.i.d. additive.


In [ ]:
u = rng.standard_cauchy(n)
Y_c = np.cumsum(u)
Q_c = make_jet(Y_c)
M_c = fit_M(Q_c)
print('Cauchy random walk, M_hat:')
print(np.round(M_c, 4))
print(f'||M_c - M_iid||_F = {np.linalg.norm(M_c - M_iid):.5f}')
print('\nNote: M_hat does not flinch on heavy tails. The Cauchy tail lives in the')
print('RESIDUAL cloud, not in the operator.')


Look at where the Cauchy tail lives:


In [ ]:
eps_c = Q_c[1:] - Q_c[:-1] @ M_c.T
print('residual cloud, Cauchy walk (level column):')
print(f'  median absolute value: {np.median(np.abs(eps_c[:,1])):.3f}')
print(f'  99th percentile:        {np.percentile(np.abs(eps_c[:,1]), 99):.3f}')
print(f'  max:                    {np.max(np.abs(eps_c[:,1])):.1f}')
print('The Cauchy tail is operationally read off the residual.')


## 4.  AR(1) Gaussian — memory enters the operator

$X_{t+1} = 0.7\, X_t + z_t$ with $z_t \sim \mathcal{N}(0, 1)$. Now the past predicts the next step. The operator moves off $\hat M_{\mathrm{iid}}$.


In [ ]:
phi = 0.7
X = np.zeros(n)
for t in range(1, n):
    X[t] = phi * X[t-1] + rng.standard_normal()
Q_ar = make_jet(X)
M_ar = fit_M(Q_ar)
print(f'AR(1) phi={phi}, M_hat:')
print(np.round(M_ar, 4))
print(f'||M_ar - M_iid||_F = {np.linalg.norm(M_ar - M_iid):.4f}')
print(f'scalar trace s_0 = {np.trace(M_ar)/3:.4f} (M_iid gives 1/3)')


The AR(1) operator has moved measurably off $\hat M_{\mathrm{iid}}$. The displacement is the substantive content of the process — the autoregressive coefficient $\phi$ is geometrically recoverable.

## 5.  Summary


In [ ]:
rows = [
    ('fair coin, p=0.5',    'iid', np.linalg.norm(np.zeros((3,3)))),  # zero ref
    ('biased coin, p=0.3',  'iid', np.linalg.norm(M_b - M_iid)),
    ('Gaussian walk',       'iid', np.linalg.norm(M_g - M_iid)),
    ('Cauchy walk',         'iid', np.linalg.norm(M_c - M_iid)),
    ('AR(1), phi=0.7',      'dep', np.linalg.norm(M_ar - M_iid)),
]
print(f'{"process":<24s}  {"class":<5s}  || M_hat - M_iid ||_F')
print('-' * 56)
for r in rows:
    print(f'{r[0]:<24s}  {r[1]:<5s}  {r[2]:.4f}')

print('\nThe four i.i.d. processes share the operator within sampling noise.')
print('The AR(1) process sits at a different operator coordinate.')
print('Memory is in the operator. The step distribution is in the residual.')


## What this teaches

- The operator $\hat M$ collapses to the same matrix $\hat M_{\mathrm{iid}}$ for *any* independent-step trajectory, regardless of whether the step has finite moments. Tail heaviness does not enter the operator.
- The residual cloud carries the step distribution. Cauchy tails, Gaussian tails, Bernoulli ±1 — all read directly off the residuals.
- AR(1) memory is a *measurable displacement* of the operator in coefficient space. The autoregressive coefficient is geometrically recoverable from the trajectory.

The same machinery handles the Cauchy walk and the AR(1) without modification. The substrate reads two layers — operator and residual — and both layers stay finite where moments do not.
